# Model Definition and Evaluation
## Table of Contents
1. [Model Selection](#model-selection)
2. [Feature Engineering](#feature-engineering)
3. [Hyperparameter Tuning](#hyperparameter-tuning)
4. [Implementation](#implementation)
5. [Evaluation Metrics](#evaluation-metrics)
6. [Comparative Analysis](#comparative-analysis)


## Model Selection

**1. Linear IV regression**
Two-Stage least squares (2SLS) regression analysis is the most widely used IV estimator. 

**Stage 1: Estimate  $X$ using the instrument $Z$**

First, run the OLS regression to get coefficients:
\begin{equation}
X = \alpha_0 + \alpha_1 Z + v \tag{1}
\end{equation}

Then, generate the predicted values (omitting the error term):
\begin{equation}
\hat{X} = \hat{\alpha}_0 + \hat{\alpha}_1 Z \tag{2}
\end{equation}

**Stage 2: Regress $Y$ on the predicted $\hat{X}$** 
Now substitute $\hat{X}$ into main equation:
\begin{equation}
Y = \beta_0 + \beta_1 \hat{X} + u \tag{3}
\end{equation}

Unknown regression coefficients: $\alpha, \beta$

Error term: $u, v$

**2. Non-linear ML-based models**

Non-linear machine-learning based models are not implemented on the datasets, concerning that ML models work best with high-dimensional data and non-linearity. Variables in our datasets show clear linear relationships and have 4-5 variables per dataset, ML-based model is therefore not needed in this project. 

**The full pipeline of the data is shown below:**

1. Pairplot exploration: visualise structure
2. Correlation diagnostics: identify IVs and chains
3. First-stage F-stat check: instrument relevance
4. 2SLS with corrected SEs: causal effect estimation (run Anderson-Rubin test on weak instrument)
5. Direction test + placebo: confirm causal arrow
6. OLS comparison: quantify endogeneity bias
7. Sensitivity analysis: robustness to unmeasured confounding (Imbens partial for valid IVs and Oster sensitivity on OLS-only data)
8. Control function (CFA): formal Hausman endogeneity test  


## Feature Engineering

No additional feature engineering performed beyond what was done for the baseline model.


## Hyperparameter Tuning

No hyperparameter tuning methods applied.


## Implementation

Implement the final model(s).


In [ ]:
"""
**Overview:**
  STEP 1 — Pairplot exploration          : visualise structure
  STEP 2 — Correlation diagnostics       : identify IVs and chains
  STEP 3 — First-stage F-stat check      : instrument relevance
            └── Anderson-Rubin CI        : (conditional, weak IV only)
  STEP 4 — 2SLS with corrected SEs       : causal effect estimation
  STEP 5 — Direction test + placebo      : confirm causal arrow
  STEP 6 — OLS comparison                : quantify endogeneity bias
  STEP 7 — Sensitivity analysis          : Imbens (IV chains)
                                           Oster δ (OLS-only chains)
  STEP 8 — Control Function (CFA)        : formal endogeneity test

Datasets:
    data      — causal_direction_iv_sem.csv
    data_ct   — clinical_trial_sem.csv
    data_ecom — ecommerce_sem.csv
    data_env  — environment_sem.csv
    data_mkt  — marketing_sem.csv
"""
# ============================================ 
# Import necessary libraries 
# ============================================
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, mean_squared_error, classification_report
import scipy as sp
import seaborn as sn
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

# ============================================ 
# Set shared Palette
# ============================================
C = {
    'iv':      '#1D9E75',   # teal    — valid instrument
    'treat':   '#7F77DD',   # purple  — treatment / mediator
    'outcome': '#D85A30',   # coral   — outcome
    'direct':  '#639922',   # green   — direct OLS treatment
    'failed':  '#888780',   # gray    — failed IV
    'degen':   '#BA7517',   # amber   — degenerate IV
    'ols':     '#7F77DD',   # purple  — OLS estimate
    'iv_est':  '#1D9E75',   # teal    — 2SLS estimate
    'warn':    '#BA7517',   # amber   — warning
    'bg':      '#FAFAF8',
    'grid':    '#D3D1C7',
    'text':    '#2C2C2A',
    'muted':   '#5F5E5A',
    'edge':    '#444441',
    'null':    '#B4B2A9',
}
 
NODE_STYLE = dict(node_size=4000, font_size=9, font_weight='bold', font_color='white')

# ============================================ 
# Data loading
# ============================================
base_path = "/kaggle/input/datasets/cloverchen/causalpitfalls-benchmark-causal-data-neurips-2025/CausalPitfallsData/causal_direction_iv"

# Define the dataset file names
files = {
    "data":      "causal_direction_iv_sem.csv",
    "data_ct":   "clinical_trial_sem.csv",
    "data_ecom": "ecommerce_sem.csv",
    "data_env":  "environment_sem.csv",
    "data_mkt":  "marketing_sem.csv",
}

# Load datasets into a dictionary
datasets = {}
for var_name, filename in files.items():
    full_path = f"{base_path}/{filename}"
    datasets[var_name] = pd.read_csv(full_path)

# Assign datasets to variables for easier access
data      = datasets["data"]
data_ct   = datasets["data_ct"]
data_ecom = datasets["data_ecom"]
data_env  = datasets["data_env"]
data_mkt  = datasets["data_mkt"]

# Create a dictionary for easy access to datasets
datasets_dict = {
    'data':      data,
    'data_ct':   data_ct,
    'data_ecom': data_ecom,
    'data_env':  data_env,
    'data_mkt':  data_mkt,
}



# ======= Perform 2SLS on the toy benchmark dataset =======
# First stage: regress X_1 on instruments Z_1, Z_2
Z = sm.add_constant(data[['Z_1', 'Z_2']])
first_stage = sm.OLS(data['X_1'], Z).fit()
X1_hat = first_stage.fittedvalues  # predicted X_1

# Second stage: regress X_2 on predicted X_1
X1_hat_const = sm.add_constant(X1_hat)
second_stage = sm.OLS(data['X_2'], X1_hat_const).fit()
print(second_stage.summary())

# First-stage F-stat (> 10 for strong instruments)
print(f"\nFirst-stage F-statistic: {first_stage.fvalue:.2f}")

# ======= Run corrected standard errors for 2SLS =======
# First stage
Z = sm.add_constant(data[['Z_1', 'Z_2']])
first_stage = sm.OLS(data['X_1'], Z).fit()
X1_hat = first_stage.fittedvalues

# Second stage
X1_hat_const = sm.add_constant(X1_hat)
second_stage = sm.OLS(data['X_2'], X1_hat_const).fit()

# --- Corrected standard errors ---
n = len(data)
X2 = data['X_2'].values
X1_actual = sm.add_constant(data['X_1'])  # use ACTUAL X_1, not predicted

# True 2SLS residuals use actual X_1
beta = second_stage.params.values
resid_corrected = X2 - X1_actual @ beta
sigma2 = np.sum(resid_corrected**2) / (n - 2)

# Corrected variance of beta
XtX_inv = np.linalg.inv(X1_hat_const.T @ X1_hat_const)
var_beta = sigma2 * XtX_inv
se_corrected = np.sqrt(np.diag(var_beta))

print(f"\n{'=' * 70}")
print("Corrected standard errors for 2SLS:")
print(f"\n{'=' * 70}")
print("Corrected IV coefficient:", beta[1])
print("Corrected std error:     ", se_corrected[1])
print("Corrected t-stat:        ", beta[1] / se_corrected[1])

# ======= Reverse 2SLS: Can Z_1, Z_2 instrument X_1 to predict X_2? ======
first_stage_rev = sm.OLS(data['X_2'], Z).fit()
X2_hat = first_stage_rev.fittedvalues

X2_hat_const = sm.add_constant(X2_hat)
second_stage_rev = sm.OLS(data['X_1'], X2_hat_const).fit()

print(f"Reverse F-statistic: {first_stage_rev.fvalue:.2f}")
print(second_stage_rev.summary())

# ======= test the symmetry of the coefficients =======
coef_forward = 0.9318   # X_1 → X_2
coef_reverse = 0.4668   # X_2 → X_1

# In a true causal system X_1 → X_2 with coefficient β:
# Forward IV should estimate β ≈ true effect
# Reverse IV should estimate 1/β only if reverse were true
print(f"Forward coefficient:        {coef_forward:.4f}")
print(f"Reverse coefficient:        {coef_reverse:.4f}")
print(f"Product (should ≈ 1 if symmetric): {coef_forward * coef_reverse:.4f}")
print(f"1/forward (expected reverse if causal): {1/coef_forward:.4f}")

# ======= Check how much variance each instrument explains in X_1 vs X_2 =======
Z = sm.add_constant(data[['Z_1', 'Z_2']])

# Check how much variance each instrument explains in X_1 vs X_2
fs_x1 = sm.OLS(data['X_1'], Z).fit()
fs_x2 = sm.OLS(data['X_2'], Z).fit()

print(f"R² of Z → X_1: {fs_x1.rsquared:.4f}  (F={fs_x1.fvalue:.2f})")
print(f"R² of Z → X_2: {fs_x2.rsquared:.4f}  (F={fs_x2.fvalue:.2f})")
print()
print("If Z explains X_1 much better than X_2,")
print("then X_1 is the treatment and X_2 is the outcome → X_1 → X_2")

#  ===== Visualize the causal graph =======
G = nx.DiGraph()
G.add_edges_from([
    ("Z_1", "X_1"),
    ("Z_2", "X_1"),
    ("X_1", "X_2")
])

pos = nx.spring_layout(G)
nx.draw(G, pos, with_labels=True, arrows=True)
plt.show()

### Running Correlation test on all 4 datasets 

Goal: 

1. Confirm perfect correlations (e.g. rainfall/soil_quality)
2. Flag instruments via the uniform distribution heuristic
3. Guide which variable to use as IV in 2SLS for each dataset

In [ ]:
# ======= Explore other datasets =======
datasets = {
    'clinical_trial': data_ct,
    'ecommerce': data_ecom,
    'environment': data_env,
    'marketing': data_mkt
}

for name, df in datasets.items():
    print(f"\n{'='*50}")
    print(f"=== {name.upper()} ===")
    print(f"{'='*50}")
    
    # Step 1: Correlation matrix to spot perfect linear pairs
    print("\n--- Correlation Matrix ---")
    print(df.corr().round(3))
    
    # Step 2: Identify likely IVs (uniform histogram = low variance relative to range)
    print("\n--- Variable distributions (std/range ratio) ---")
    for col in df.columns:
        ratio = df[col].std() / (df[col].max() - df[col].min())
        tag = "← likely IV (uniform)" if ratio < 0.25 else ""
        print(f"  {col}: std/range = {ratio:.3f} {tag}")
        
# ===== Define functions for 2SLS and OLS =======
def run_2sls(df, instruments, treatment, outcome, dataset_name, chain_name):
    print(f"\n{'='*55}")
    print(f"  {dataset_name} | {chain_name}")
    print(f"  IV: {instruments} → {treatment} → {outcome}")
    print(f"{'='*55}")
    
    Z = sm.add_constant(df[instruments])
    X = df[treatment]
    Y = df[outcome]
    
    # --- First Stage ---
    first_stage = sm.OLS(X, Z).fit()
    X_hat = first_stage.fittedvalues
    
    print(f"\n[First Stage] {instruments} → {treatment}")
    print(f"  R²     : {first_stage.rsquared:.4f}")
    print(f"  F-stat : {first_stage.fvalue:.2f}  {'Strong' if first_stage.fvalue > 10 else 'Weak'} instrument")
    
    # --- Second Stage ---
    X_hat_const = sm.add_constant(X_hat)
    second_stage = sm.OLS(Y, X_hat_const).fit()
    
    # --- Corrected Standard Errors ---
    beta = second_stage.params.values
    X_actual = sm.add_constant(X)
    resid_corrected = Y.values - X_actual.values @ beta
    sigma2 = np.sum(resid_corrected**2) / (len(Y) - 2)
    XtX_inv = np.linalg.inv(X_hat_const.values.T @ X_hat_const.values)
    se_corrected = np.sqrt(np.diag(sigma2 * XtX_inv))
    t_stat = beta[1] / se_corrected[1]
    
    print(f"\n[Second Stage] {treatment} → {outcome}")
    print(f"  IV Coefficient : {beta[1]:.4f}")
    print(f"  Corrected SE   : {se_corrected[1]:.4f}")
    print(f"  t-statistic    : {t_stat:.4f}")
    print(f"  Significant?   : {'Yes (|t|>1.96)' if abs(t_stat) > 1.96 else 'No'}")
    
    # --- Exclusion Restriction Check ---
    fs_outcome = sm.OLS(Y, Z).fit()
    print(f"\n[Exclusion Check]")
    print(f"  R² IV → {treatment} (should be HIGH) : {first_stage.rsquared:.4f}")
    print(f"  R² IV → {outcome} directly (LOW=good): {fs_outcome.rsquared:.4f}")
    ratio = first_stage.rsquared / max(fs_outcome.rsquared, 0.0001)
    print(f"  Ratio (>3 = exclusion likely holds)  : {ratio:.2f}  {'Strong' if ratio > 3 else 'Weak exclusion'}")
    
    return beta[1]

# ===== Define function for OLS (direct effect) =======
def run_ols(df, treatment, outcome, dataset_name, chain_name):
    """For direct causal paths where no IV is needed."""
    print(f"\n{'='*55}")
    print(f"  {dataset_name} | {chain_name} [OLS - Direct Effect]")
    print(f"{'='*55}")
    
    Z = sm.add_constant(df[treatment])
    model = sm.OLS(df[outcome], Z).fit()
    
    print(f"  Coefficient : {model.params[1]:.4f}")
    print(f"  Std Error   : {model.bse[1]:.4f}")
    print(f"  t-statistic : {model.tvalues[1]:.4f}")
    print(f"  R²          : {model.rsquared:.4f}")
    print(f"  Significant?: {'Yes' if model.pvalues[1] < 0.05 else 'No'}")
    return model.params[1]

# ════════════════════════════════════════════════════
# 1. CLINICAL TRIAL
# ════════════════════════════════════════════════════
# baseline_health is IV (r≈0.03 with all others = exogenous)
# dosage_mg and drug_concentration are near-identical (r=0.998)
# Run both to confirm they're statistically inseparable

print("\n" + "█"*55)
print("█  CLINICAL TRIAL")
print("█"*55)

coef_a = run_2sls(data_ct,
    instruments=['baseline_health'],
    treatment='dosage_mg',
    outcome='health_improvement',
    dataset_name='Clinical Trial',
    chain_name='baseline_health(IV) → dosage_mg → health_improvement')

coef_b = run_2sls(data_ct,
    instruments=['baseline_health'],
    treatment='drug_concentration',
    outcome='health_improvement',
    dataset_name='Clinical Trial',
    chain_name='baseline_health(IV) → drug_concentration → health_improvement')

print(f"\n[Mediator Comparison]")
print(f"  Via dosage_mg        : {coef_a:.4f}")
print(f"  Via drug_concentration: {coef_b:.4f}")
print(f"  Difference           : {abs(coef_a - coef_b):.4f}")
print(f"  Conclusion: {'Statistically inseparable — domain knowledge needed' if abs(coef_a-coef_b) < 0.05 else 'Different mediators!'}")

# ════════════════════════════════════════════════════
# 2. E-COMMERCE (two independent chains)
# ════════════════════════════════════════════════════
# Chain 1: age(IV) → visits → ? (visits and age r=0.996, purchases unrelated)
# Chain 2: income(IV) → purchases (direct, r=0.999)
# Key question: does visits cause purchases?

print("\n" + "█"*55)
print("█  E-COMMERCE")
print("█"*55)

# Chain 1: age as IV for visits effect on purchases
run_2sls(data_ecom,
    instruments=['age'],
    treatment='visits',
    outcome='purchases',
    dataset_name='E-commerce',
    chain_name='age(IV) → visits → purchases')

# Chain 2: income direct effect on purchases (OLS, income IS the driver)
run_ols(data_ecom,
    treatment='income',
    outcome='purchases',
    dataset_name='E-commerce',
    chain_name='income → purchases')

# Placebo test: cross-chain (should show NO effect)
print("\n[Placebo Tests — cross-chain effects should be ~0]")
run_2sls(data_ecom,
    instruments=['age'],
    treatment='visits',
    outcome='income',
    dataset_name='E-commerce',
    chain_name='PLACEBO: age(IV) → visits → income (expect ~0)')

# ════════════════════════════════════════════════════
# 3. ENVIRONMENT
# ════════════════════════════════════════════════════
# rainfall(IV) → soil_quality → crop_yield  [rainfall r=0.999 with soil]
# fertilizer → crop_yield  [direct treatment, r=0.919, use OLS]
# rainfall NOT a valid IV for fertilizer chain (r=0.081 with fertilizer)

print("\n" + "█"*55)
print("█  ENVIRONMENT")
print("█"*55)

# Chain 1: rainfall as IV for soil_quality effect on crop_yield
run_2sls(data_env,
    instruments=['rainfall_mm'],
    treatment='soil_quality',
    outcome='crop_yield',
    dataset_name='Environment',
    chain_name='rainfall(IV) → soil_quality → crop_yield')

# Chain 2: fertilizer direct effect (OLS — no valid IV available)
run_ols(data_env,
    treatment='fertilizer_kg',
    outcome='crop_yield',
    dataset_name='Environment',
    chain_name='fertilizer_kg → crop_yield (direct)')

# Combined OLS: both drivers together
print("\n[Combined Model: fertilizer + soil_quality → crop_yield]")
Z_combined = sm.add_constant(data_env[['fertilizer_kg', 'soil_quality']])
combined = sm.OLS(data_env['crop_yield'], Z_combined).fit()
print(combined.summary().tables[1])

# ════════════════════════════════════════════════════
# 4. MARKETING
# ════════════════════════════════════════════════════
# ad_spend(IV) → brand_awareness → sales  [r=0.999, near identical]
# promo_spend → sales  [direct driver, r=0.777, use OLS]
# brand_awareness may just be ad_spend renamed (r=0.999)

print("\n" + "█"*55)
print("█  MARKETING")
print("█"*55)

# Chain 1: ad_spend as IV for brand_awareness effect on sales
run_2sls(data_mkt,
    instruments=['ad_spend_k'],
    treatment='brand_awareness',
    outcome='sales',
    dataset_name='Marketing',
    chain_name='ad_spend(IV) → brand_awareness → sales')

# Chain 2: promo_spend direct effect on sales
run_ols(data_mkt,
    treatment='promo_spend_k',
    outcome='sales',
    dataset_name='Marketing',
    chain_name='promo_spend_k → sales (direct)')

# Degeneracy check: are ad_spend and brand_awareness the same construct?
print("\n[Degeneracy Check: ad_spend vs brand_awareness]")
print(f"  Correlation: {data_mkt['ad_spend_k'].corr(data_mkt['brand_awareness']):.4f}")
print(f"  If > 0.99 → they are statistically the same variable")
print(f"  Implication: 2SLS above is instrumenting X with itself — treat with caution")

# Safer alternative: use ad_spend directly
run_ols(data_mkt,
    treatment='ad_spend_k',
    outcome='sales',
    dataset_name='Marketing',
    chain_name='ad_spend_k → sales (treating as direct treatment)')

In [ ]:
# ======= Run simple OLS on 'Clinical Trial' dataset as baseline_health is not a valid IV for dosage_mg or drug_concentration. =======

# No valid IV exists in "Clinical Trial" dataset for 2SLS
# Fall back to OLS and acknowledge the limitation
print("\n[Clinical Trial — OLS fallback]")

Z = sm.add_constant(data_ct['dosage_mg'])
ols_dose = sm.OLS(data_ct['health_improvement'], Z).fit()
print(f"OLS: dosage_mg → health_improvement: {ols_dose.params[1]:.4f} (R²={ols_dose.rsquared:.4f})")

Z2 = sm.add_constant(data_ct['drug_concentration'])
ols_conc = sm.OLS(data_ct['health_improvement'], Z2).fit()
print(f"OLS: drug_concentration → health_improvement: {ols_conc.params[1]:.4f} (R²={ols_conc.rsquared:.4f})")

# Note:
# baseline_health fails the relevance assumption (F=0.86 < 10).
# No valid IV identified. OLS estimates are reported but may be
# confounded. Causal interpretation requires external instrument.

In [ ]:
# ============================================ 
# Dataset configurations
# Defined after Steps 2-3 established:
#   - Which variables are valid instruments (Step 2)
#   - Whether each instrument is strong (Step 3, F > 10)
#   - Causal direction (confirmed later in Step 5)
# ============================================

CONFIGS = [
    {
        'name':        'causal_direction_iv',
        'df_var':      'data',
        'instruments': ['Z_1', 'Z_2'],
        'treatment':   'X_1',
        'outcome':     'X_2',
        'iv_valid':    True,
        'note':        'Both IVs valid (F=168.7)',
    },
    {
        'name':        'clinical_trial',
        'df_var':      'data_ct',
        'instruments': ['baseline_health'],
        'treatment':   'dosage_mg',
        'outcome':     'health_improvement',
        'iv_valid':    False,               # F = 0.86 — weak IV
        'note':        'IV failed (F=0.86) — OLS fallback + AR test',
    },
    {
        'name':        'ecommerce — income chain',
        'df_var':      'data_ecom',
        'instruments': None,               # direct OLS only
        'treatment':   'income',
        'outcome':     'purchases',
        'iv_valid':    False,
        'note':        'Direct OLS (income→purchases, R²=0.998)',
    },
    {
        'name':        'environment — soil chain',
        'df_var':      'data_env',
        'instruments': ['rainfall_mm'],
        'treatment':   'soil_quality',
        'outcome':     'crop_yield',
        'iv_valid':    True,
        'note':        'rainfall_mm valid IV (F=859,535)',
    },
    {
        'name':        'marketing — promo chain',
        'df_var':      'data_mkt',
        'instruments': None,               # promo is direct driver
        'treatment':   'promo_spend_k',
        'outcome':     'sales',
        'iv_valid':    False,
        'note':        'Direct OLS (ad_spend IV degenerate — not used)',
    },
]
 
# OLS chains without valid IV — used for Oster sensitivity
OLS_ONLY_CONFIGS = [
    {'name': 'clinical_trial (dosage)',     'df': data_ct,   'treatment': 'dosage_mg',      'outcome': 'health_improvement'},
    {'name': 'clinical_trial (drug_conc)',  'df': data_ct,   'treatment': 'drug_concentration', 'outcome': 'health_improvement'},
    {'name': 'ecommerce (income)',          'df': data_ecom, 'treatment': 'income',          'outcome': 'purchases'},
    {'name': 'environment (fertilizer)',    'df': data_env,  'treatment': 'fertilizer_kg',   'outcome': 'crop_yield'},
    {'name': 'marketing (promo_spend)',     'df': data_mkt,  'treatment': 'promo_spend_k',   'outcome': 'sales'},
]
 
# All chains including additional OLS-only ones not in CONFIGS
ALL_OLS_CONFIGS = [
    {'name': 'causal_direction_iv',           'df': data,      'treatment': 'X_1',               'outcome': 'X_2'},
    {'name': 'clinical_trial (dosage)',        'df': data_ct,   'treatment': 'dosage_mg',          'outcome': 'health_improvement'},
    {'name': 'clinical_trial (drug_conc)',     'df': data_ct,   'treatment': 'drug_concentration', 'outcome': 'health_improvement'},
    {'name': 'ecommerce (visits)',             'df': data_ecom, 'treatment': 'visits',             'outcome': 'purchases'},
    {'name': 'ecommerce (income)',             'df': data_ecom, 'treatment': 'income',             'outcome': 'purchases'},
    {'name': 'environment (soil)',             'df': data_env,  'treatment': 'soil_quality',       'outcome': 'crop_yield'},
    {'name': 'environment (fertilizer)',       'df': data_env,  'treatment': 'fertilizer_kg',      'outcome': 'crop_yield'},
    {'name': 'marketing (brand_awareness)',    'df': data_mkt,  'treatment': 'brand_awareness',    'outcome': 'sales'},
    {'name': 'marketing (promo_spend)',        'df': data_mkt,  'treatment': 'promo_spend_k',      'outcome': 'sales'},
]
 

### Summary 

In [ ]:
# ======= Summarize all findings cleanly for notebook =======
results = {
    'Clinical Trial':  {'IV': 'None (baseline_health too weak)', 
                        'direction': 'Unknown', 'effect': 'N/A'},
    'Ecom: visits':    {'IV': 'age', 
                        'direction': 'visits ✗→ purchases', 'effect': 'None'},
    'Ecom: purchases': {'IV': 'N/A (OLS)', 
                        'direction': 'income → purchases', 'effect': 0.010},
    'Environment:soil':{'IV': 'rainfall', 
                        'direction': 'soil → crop_yield', 'effect': 0.9475},
    'Environment:fert':{'IV': 'N/A (OLS)', 
                        'direction': 'fertilizer → crop_yield', 'effect': 1.242},
    'Marketing:ad':    {'IV': 'Degenerate (ad≈brand)', 
                        'direction': 'ad/brand → sales', 'effect': 0.528},
    'Marketing:promo': {'IV': 'N/A (OLS)', 
                        'direction': 'promo → sales', 'effect': 1.285},
}

print(f"\n{'Dataset':<22} {'IV Status':<30} {'Direction':<25} {'Effect'}")
print("─"*90)
for k, v in results.items():
    print(f"{k:<22} {v['IV']:<30} {v['direction']:<25} {v['effect']}")

# ======= Visualize causal graphs for all datasets =======
"""
Causal IV Analysis — Causal Graph Visualizations
=================================================
Renders causal DAGs for all 5 datasets using networkx + matplotlib.
Run each cell independently or run all at once.

Node color legend:
  Teal   — Instrumental variable (IV)
  Purple — Treatment / mediator
  Coral  — Outcome
  Green  — Direct-effect treatment (OLS, no IV needed)
  Gray   — Failed / degenerate IV
"""

# ── Shared style constants ────────────────────────────────────────────────────
COLORS = {
    'iv':       '#1D9E75',   # teal   — valid instrument
    'treat':    '#7F77DD',   # purple — treatment / mediator
    'outcome':  '#D85A30',   # coral  — outcome variable
    'direct':   '#639922',   # green  — direct-effect treatment (OLS)
    'failed':   '#888780',   # gray   — failed / degenerate IV
    'degen':    '#BA7517',   # amber  — degenerate IV (marketing)
    'bg':       '#FAFAF8',
    'edge':     '#444441',
    'edge_null':'#B4B2A9',
}

NODE_STYLE = dict(node_size=4000, font_size=9, font_weight='bold', font_color='white')

def add_edge_label(ax, pos, u, v, label, color='#444441', offset=(0, 0.08)):
    # Place a label at the midpoint of an edge.
    x = (pos[u][0] + pos[v][0]) / 2 + offset[0]
    y = (pos[u][1] + pos[v][1]) / 2 + offset[1]
    ax.text(x, y, label, ha='center', va='center', fontsize=8,
            color=color, style='italic',
            bbox=dict(boxstyle='round,pad=0.2', fc='white', ec='none', alpha=0.8))

def draw_null_edge(ax, pos, u, v, label='no effect'):
    # Draw a dashed gray arrow for a non-significant path.
    x0, y0 = pos[u]
    x1, y1 = pos[v]
    ax.annotate('', xy=(x1, y1), xytext=(x0, y0),
                arrowprops=dict(arrowstyle='->', color=COLORS['edge_null'],
                                lw=1.2, linestyle='dashed',
                                connectionstyle='arc3,rad=0.0'))
    mx, my = (x0+x1)/2, (y0+y1)/2
    ax.text(mx, my + 0.1, label, ha='center', fontsize=7.5,
            color=COLORS['edge_null'], style='italic')

def style_ax(ax, title, subtitle=''):
    ax.set_title(title, fontsize=12, fontweight='bold', pad=14, color='#2C2C2A')
    if subtitle:
        ax.text(0.5, -0.04, subtitle, transform=ax.transAxes,
                ha='center', fontsize=8, color='#5F5E5A', style='italic')
    ax.set_facecolor(COLORS['bg'])
    ax.axis('off')

# ══════════════════════════════════════════════════════════════════════════════
# 1.  causal_direction_iv benchmark
# ══════════════════════════════════════════════════════════════════════════════
def plot_causal_direction_iv(ax=None):
    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=(7, 3.5))
        fig.patch.set_facecolor(COLORS['bg'])

    G = nx.DiGraph()
    G.add_nodes_from(['Z_1 (IV)', 'Z_2 (IV)', 'X_1', 'X_2'])
    G.add_edges_from([('Z_1 (IV)', 'X_1'), ('Z_2 (IV)', 'X_1'), ('X_1', 'X_2')])

    pos = {'Z_1 (IV)': (0, 0.6), 'Z_2 (IV)': (0, 0),
           'X_1': (1.5, 0.3), 'X_2': (3.0, 0.3)}

    node_colors = [COLORS['iv'], COLORS['iv'], COLORS['treat'], COLORS['outcome']]

    nx.draw_networkx(G, pos, ax=ax, node_color=node_colors,
                     edge_color=COLORS['edge'], arrows=True,
                     arrowsize=20, width=1.8, **NODE_STYLE)

    add_edge_label(ax, pos, 'X_1', 'X_2', 'β = 0.93\nt = 14.2', offset=(0, 0.15))
    add_edge_label(ax, pos, 'Z_1 (IV)', 'X_1', 'F = 168.7', offset=(0, 0.12))

    style_ax(ax, 'causal_direction_iv',
             'Both IVs valid  ·  Confirmed direction: X_1 → X_2')

    if standalone:
        plt.tight_layout()
        plt.savefig('graph_causal_direction_iv.png', dpi=150, bbox_inches='tight')
        plt.show()

# ══════════════════════════════════════════════════════════════════════════════
# 2.  clinical_trial
# ══════════════════════════════════════════════════════════════════════════════
def plot_clinical_trial(ax=None):
    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=(8, 3.5))
        fig.patch.set_facecolor(COLORS['bg'])

    G = nx.DiGraph()
    nodes = ['baseline_health\n(IV)', 'dosage_mg', 'drug_conc', 'health_improv.']
    G.add_nodes_from(nodes)
    G.add_edges_from([
        ('baseline_health\n(IV)', 'dosage_mg'),
        ('baseline_health\n(IV)', 'drug_conc'),
        ('dosage_mg', 'drug_conc'),
        ('dosage_mg', 'health_improv.'),
        ('drug_conc', 'health_improv.'),
    ])

    pos = {
        'baseline_health\n(IV)': (0, 0.3),
        'dosage_mg':               (1.5, 0.7),
        'drug_conc':               (1.5, 0.0),
        'health_improv.':          (3.2, 0.3),
    }

    node_colors = [COLORS['failed'], COLORS['treat'], COLORS['treat'], COLORS['outcome']]

    nx.draw_networkx(G, pos, ax=ax, node_color=node_colors,
                     edge_color=COLORS['edge'], arrows=True,
                     arrowsize=18, width=1.5, **NODE_STYLE)

    # Annotate the failed IV
    bh_x, bh_y = pos['baseline_health\n(IV)']
    ax.text(bh_x, bh_y - 0.25, 'F = 0.86  (need > 10)',
            ha='center', fontsize=7.5, color='#A32D2D',
            bbox=dict(boxstyle='round,pad=0.3', fc='#FCEBEB', ec='#F09595', lw=0.8))

    add_edge_label(ax, pos, 'dosage_mg', 'drug_conc', 'r = 0.998', offset=(0.15, 0.05))
    add_edge_label(ax, pos, 'dosage_mg', 'health_improv.', 'r = 0.992', offset=(0, 0.14))
    add_edge_label(ax, pos, 'drug_conc', 'health_improv.', 'r = 0.996', offset=(0, -0.14))

    style_ax(ax, 'clinical_trial',
             'baseline_health fails relevance (F=0.86)  ·  OLS fallback only  ·  dosage ≈ drug_conc (r=0.998)')

    if standalone:
        plt.tight_layout()
        plt.savefig('graph_clinical_trial.png', dpi=150, bbox_inches='tight')
        plt.show()


# ══════════════════════════════════════════════════════════════════════════════
# 3.  ecommerce
# ══════════════════════════════════════════════════════════════════════════════
def plot_ecommerce(ax=None):
    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=(8, 4.5))
        fig.patch.set_facecolor(COLORS['bg'])

    # Chain 1: age → visits ✗→ purchases
    G = nx.DiGraph()
    G.add_nodes_from(['age\n(IV)', 'visits', 'income\n(direct)', 'purchases'])
    G.add_edges_from([('age\n(IV)', 'visits'), ('income\n(direct)', 'purchases')])

    pos = {
        'age\n(IV)':       (0, 1.0),
        'visits':          (1.5, 1.0),
        'income\n(direct)':(0, 0.0),
        'purchases':       (1.5, 0.0),
    }

    node_colors = [COLORS['iv'], COLORS['treat'], COLORS['direct'], COLORS['outcome']]
    nx.draw_networkx(G, pos, ax=ax, node_color=node_colors,
                     edge_color=COLORS['edge'], arrows=True,
                     arrowsize=20, width=2.0, **NODE_STYLE)

    # Null edge: visits ✗→ purchases
    draw_null_edge(ax, pos, 'visits', 'purchases', 'no effect (t=1.72)')

    add_edge_label(ax, pos, 'age\n(IV)', 'visits', 'r=0.996\nF=124,406', offset=(0, 0.12))
    add_edge_label(ax, pos, 'income\n(direct)', 'purchases',
                   'β=0.01/$\nR²=0.998', offset=(0, -0.14))

    # Chain labels
    ax.text(-0.3, 1.25, 'Chain 1', fontsize=8, color='#5F5E5A', style='italic')
    ax.text(-0.3, 0.25, 'Chain 2', fontsize=8, color='#5F5E5A', style='italic')
    ax.axhline(0.5, color=COLORS['edge_null'], lw=0.5, linestyle=':')

    style_ax(ax, 'ecommerce',
             'Two independent chains  ·  visits ✗→ purchases (null)  ·  income → purchases (R²=0.998)')

    if standalone:
        plt.tight_layout()
        plt.savefig('graph_ecommerce.png', dpi=150, bbox_inches='tight')
        plt.show()

# ══════════════════════════════════════════════════════════════════════════════
# 4.  environment
# ══════════════════════════════════════════════════════════════════════════════
def plot_environment(ax=None):
    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=(8, 4.0))
        fig.patch.set_facecolor(COLORS['bg'])

    G = nx.DiGraph()
    nodes = ['rainfall_mm\n(IV)', 'soil_quality', 'fertilizer_kg\n(direct)', 'crop_yield']
    G.add_nodes_from(nodes)
    G.add_edges_from([
        ('rainfall_mm\n(IV)', 'soil_quality'),
        ('soil_quality', 'crop_yield'),
        ('fertilizer_kg\n(direct)', 'crop_yield'),
    ])

    pos = {
        'rainfall_mm\n(IV)':    (0, 0.8),
        'soil_quality':         (1.5, 0.8),
        'fertilizer_kg\n(direct)': (1.5, 0.1),
        'crop_yield':           (3.2, 0.5),
    }

    node_colors = [COLORS['iv'], COLORS['treat'], COLORS['direct'], COLORS['outcome']]
    nx.draw_networkx(G, pos, ax=ax, node_color=node_colors,
                     edge_color=COLORS['edge'], arrows=True,
                     arrowsize=20, width=2.0, **NODE_STYLE)

    add_edge_label(ax, pos, 'rainfall_mm\n(IV)', 'soil_quality',
                   'r=0.999\nF=859,535', offset=(0, 0.14))
    add_edge_label(ax, pos, 'soil_quality', 'crop_yield',
                   'β=0.95 (IV)\nt=16.7', offset=(0, 0.16))
    add_edge_label(ax, pos, 'fertilizer_kg\n(direct)', 'crop_yield',
                   'β=1.24 (OLS)\nt=73.7', offset=(0, -0.16))

    style_ax(ax, 'environment',
             'Two additive pathways  ·  Fertilizer effect (β=1.24) > soil quality (β=0.95)')

    if standalone:
        plt.tight_layout()
        plt.savefig('graph_environment.png', dpi=150, bbox_inches='tight')
        plt.show()


# ══════════════════════════════════════════════════════════════════════════════
# 5.  marketing
# ══════════════════════════════════════════════════════════════════════════════
def plot_marketing(ax=None):
    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=(8, 4.5))
        fig.patch.set_facecolor(COLORS['bg'])

    G = nx.DiGraph()
    nodes = ['ad_spend_k\n(degen. IV)', 'brand_awareness\n(≈ same var)', 'promo_spend_k\n(direct)', 'sales']
    G.add_nodes_from(nodes)
    G.add_edges_from([
        ('ad_spend_k\n(degen. IV)', 'brand_awareness\n(≈ same var)'),
        ('brand_awareness\n(≈ same var)', 'sales'),
        ('promo_spend_k\n(direct)', 'sales'),
    ])

    pos = {
        'ad_spend_k\n(degen. IV)':    (0, 0.85),
        'brand_awareness\n(≈ same var)':  (1.7, 0.85),
        'promo_spend_k\n(direct)':        (1.7, 0.1),
        'sales':                          (3.4, 0.5),
    }

    node_colors = [COLORS['degen'], COLORS['degen'], COLORS['direct'], COLORS['outcome']]
    nx.draw_networkx(G, pos, ax=ax, node_color=node_colors,
                     edge_color=COLORS['edge'], arrows=True,
                     arrowsize=18, width=1.8, **NODE_STYLE)

    add_edge_label(ax, pos, 'ad_spend_k\n(degen. IV)',
                   'brand_awareness\n(≈ same var)',
                   'r=0.999\ndegenerate', offset=(0, 0.16))
    add_edge_label(ax, pos, 'brand_awareness\n(≈ same var)', 'sales',
                   'β≈0.53\n(treat w/ caution)', offset=(0, 0.16))
    add_edge_label(ax, pos, 'promo_spend_k\n(direct)', 'sales',
                   'β=1.29 (OLS)\nt=39.0', offset=(0, -0.16))

    style_ax(ax, '📢  marketing',
             'ad_spend ≈ brand_awareness (r=0.999) — degenerate IV  ·  promo_spend is the reliable driver')

    if standalone:
        plt.tight_layout()
        plt.savefig('graph_marketing.png', dpi=150, bbox_inches='tight')
        plt.show()

# ══════════════════════════════════════════════════════════════════════════════
# COMBINED FIGURE — all 5 datasets in one plot
# ══════════════════════════════════════════════════════════════════════════════
def plot_all():
    fig = plt.figure(figsize=(18, 16))
    fig.patch.set_facecolor(COLORS['bg'])
    fig.suptitle('Causal IV Analysis — Causal Graphs Across All Datasets',
                 fontsize=15, fontweight='bold', color='#2C2C2A', y=0.98)

    # Grid: 3 rows x 2 cols, last slot = legend
    axes = [
        fig.add_subplot(3, 2, 1),
        fig.add_subplot(3, 2, 2),
        fig.add_subplot(3, 2, 3),
        fig.add_subplot(3, 2, 4),
        fig.add_subplot(3, 2, 5),
        fig.add_subplot(3, 2, 6),  # legend slot
    ]

    plot_causal_direction_iv(ax=axes[0])
    plot_clinical_trial(ax=axes[1])
    plot_ecommerce(ax=axes[2])
    plot_environment(ax=axes[3])
    plot_marketing(ax=axes[4])

    # Legend panel
    ax_leg = axes[5]
    ax_leg.set_facecolor(COLORS['bg'])
    ax_leg.axis('off')
    ax_leg.set_title('Node legend', fontsize=11, fontweight='bold',
                     color='#2C2C2A', pad=10)

    legend_items = [
        (COLORS['iv'],      'Instrumental variable (IV) — valid'),
        (COLORS['treat'],   'Treatment / mediator variable'),
        (COLORS['outcome'], 'Outcome variable'),
        (COLORS['direct'],  'Direct-effect treatment (OLS path)'),
        (COLORS['failed'],  'Failed IV (relevance assumption violated)'),
        (COLORS['degen'],   'Degenerate IV (r≈1 with treatment)'),
    ]
    for i, (color, label) in enumerate(legend_items):
        y = 0.82 - i * 0.13
        ax_leg.add_patch(plt.Circle((0.08, y), 0.04, color=color,
                                    transform=ax_leg.transAxes))
        ax_leg.text(0.16, y, label, transform=ax_leg.transAxes,
                    va='center', fontsize=9, color='#2C2C2A')

    # Edge legend
    ax_leg.text(0.08, 0.82 - 6*0.13 - 0.02, 'Edge types:', transform=ax_leg.transAxes,
                fontsize=9, color='#5F5E5A', fontweight='bold')
    ax_leg.annotate('', xy=(0.22, 0.82 - 7*0.13),
                    xytext=(0.06, 0.82 - 7*0.13),
                    xycoords='axes fraction',
                    arrowprops=dict(arrowstyle='->', color=COLORS['edge'], lw=1.8))
    ax_leg.text(0.25, 0.82 - 7*0.13, 'Causal path (solid)',
                transform=ax_leg.transAxes, va='center', fontsize=9)

    ax_leg.annotate('', xy=(0.22, 0.82 - 8*0.13),
                    xytext=(0.06, 0.82 - 8*0.13),
                    xycoords='axes fraction',
                    arrowprops=dict(arrowstyle='->', color=COLORS['edge_null'],
                                    lw=1.4, linestyle='dashed'))
    ax_leg.text(0.25, 0.82 - 8*0.13, 'Null / broken path (dashed)',
                transform=ax_leg.transAxes, va='center', fontsize=9)

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig('causal_graphs_all_datasets.png', dpi=150, bbox_inches='tight',
                facecolor=COLORS['bg'])
    plt.show()

# ══════════════════════════════════════════════════════════════════════════════
# RUN
# ══════════════════════════════════════════════════════════════════════════════
# Plot all in one figure 
plot_all()

# Plot individual plots (one per notebook section)
# plot_causal_direction_iv()
# plot_clinical_trial()
# plot_ecommerce()
# plot_environment()
# plot_marketing()

### 2. Non-linear (ML) methods 

Non-linear ML methods will not be implemented. The pairplots has already shown clear linear relationships, and our dataset has no high-dimentional covariates (only 4-5 variables per dataset), and we do not observe any subgroup (heterogeneous effect), thus ML methods is not necessary. 


## Evaluation Metrics

Clearly specify which metrics will be used to evaluate the model performance, and why these metrics are chosen. 

Instrument validity: 
- First-stage F-statistic > 10 (Staiger-Stock): to check whether the instrument is strong enough
- Exclusion R² ratio > 3: exclusion restriction check that IV affects outcome only through treatment
- Hausman test via CF residuals: formal endogeneity test that the treatment is truly endogenous
- Anderson-Rubin (AR) test: applied conditionally on weak instrument (clinical_trial only, F = 0.86)
    
Exclusion restriction validation: 
- Placebo tests 
- Overidentification / Sargan-Hansen (when applicable)
- Subsample / cross-chain tests
    
Estimate robustness:                  
- Imbens (2003) partial-R² sensitivity: to test how robust the causal estimate is to hidden confounders (whether unmeasured confounder could plausibly overturn the finding)
- Oster (2019) δ sensitivity: applied to OLS-only chains (clinical_trial fallback, ecommerce income, environment fertilizer, marketing promo). Reports δ: how much unobservables must matter relative to observables to explain away the OLS estimate. δ > 1 indicates robustness.

In [ ]:
# Robustness & Sensitivity Analysis
"""
Sensitivity Analysis & OLS vs 2SLS Comparison
===============================================
Two analyses for all 5 causal IV datasets:

  PART 1 — OLS vs 2SLS bias comparison
    Shows how much OLS over/underestimates the causal effect
    relative to the IV (2SLS) estimate, and why IV was needed.

  PART 2 — Sensitivity analysis (Oster / partial-R² approach for OLS-only data)
    For each valid IV result, uses the Imbens (2003) partial-R² sensitivity framework to quantify how strong an unmeasured confounder would need to be to overturn the finding

Datasets expected in scope:
    data      — causal_direction_iv_sem.csv
    data_ct   — clinical_trial_sem.csv
    data_ecom — ecommerce_sem.csv
    data_env  — environment_sem.csv
    data_mkt  — marketing_sem.csv
"""

# ── Palette ───────────────────────────────────────────────────────────────────
C = {
    'ols':    '#7F77DD',   # purple  — OLS estimate
    'iv':     '#1D9E75',   # teal    — 2SLS estimate
    'true':   '#D85A30',   # coral   — true / OLS-fallback
    'ci':     '#9FE1CB',   # light teal — confidence interval
    'warn':   '#BA7517',   # amber   — warning / degenerate
    'fail':   '#888780',   # gray    — failed IV
    'bg':     '#FAFAF8',
    'grid':   '#D3D1C7',
    'text':   '#2C2C2A',
    'muted':  '#5F5E5A',
}

# ══════════════════════════════════════════════════════════════════════════════
# HELPER FUNCTIONS
# ══════════════════════════════════════════════════════════════════════════════

def run_ols(df, treatment, outcome):
    # Return OLS coefficient, SE, p-value for treatment → outcome.
    X = sm.add_constant(df[treatment])
    res = sm.OLS(df[outcome], X).fit()
    return res.params[1], res.bse[1], res.pvalues[1], res.rsquared


def run_2sls(df, instruments, treatment, outcome):
    # Return corrected 2SLS coefficient, SE, t-stat, first-stage F.
    # Uses manual two-stage OLS with corrected residuals for proper SEs.
    
    Z = sm.add_constant(df[instruments])
    X = df[treatment]
    Y = df[outcome]

    # Stage 1
    fs = sm.OLS(X, Z).fit()
    X_hat = fs.fittedvalues

    # Stage 2
    X_hat_c = sm.add_constant(X_hat)
    ss = sm.OLS(Y, X_hat_c).fit()
    beta = ss.params.values

    # Corrected SEs using actual X residuals
    X_actual = sm.add_constant(X)
    resid = Y.values - X_actual.values @ beta
    sigma2 = np.sum(resid**2) / (len(Y) - 2)
    XtX_inv = np.linalg.inv(X_hat_c.values.T @ X_hat_c.values)
    se = np.sqrt(np.diag(sigma2 * XtX_inv))

    t_stat = beta[1] / se[1]
    return beta[1], se[1], t_stat, fs.fvalue, fs.rsquared


def sensitivity_partial_r2(df, instruments, treatment, outcome, iv_coef, iv_se,
                            n_grid=200):
    """
    Imbens (2003) partial-R² sensitivity analysis.

    Asks: how correlated would an unmeasured confounder U need to be
    with BOTH the treatment and the outcome to reduce the IV estimate
    to zero (or flip its sign)?

    Returns a grid of (r_treat, r_outcome) pairs that would overturn
    the result, plus the benchmark partial-R² of observed covariates.

    Method:
      - Compute the 'breakdown' confounding: the minimum product
        r_TU * r_YU such that the bias equals the IV coefficient.
      - Bias formula (Imbens 2003): bias ≈ (r_TU * r_YU * σ_Y) / σ_T
        where r_TU = corr(T, U|Z) and r_YU = corr(Y, U|Z).
      - Plot the contour where bias = iv_coef (the breakdown curve).
      - Benchmark: partial-R² of each observed covariate with treatment
        and outcome (to show where observed confounders sit on the plot).
    """
    Z_cols = [c for c in df.columns if c.startswith('Z')]
    X_cols = [c for c in df.columns if c.startswith('X')]

    # Residualize treatment and outcome on instruments
    Z = sm.add_constant(df[Z_cols])
    res_T = sm.OLS(df[treatment], Z).fit().resid
    res_Y = sm.OLS(df[outcome],   Z).fit().resid

    sigma_T = np.std(res_T)
    sigma_Y = np.std(res_Y)

    # Bias = r_TU * r_YU * (sigma_Y / sigma_T)
    # Breakdown: r_TU * r_YU = iv_coef * sigma_T / sigma_Y
    breakdown_product = abs(iv_coef) * sigma_T / sigma_Y

    # Grid of r_TU values [0, 1]
    r_TU = np.linspace(0.01, 0.99, n_grid)
    # r_YU needed to exactly cancel the IV estimate
    r_YU_breakdown = np.clip(breakdown_product / r_TU, 0, 1)

    # Benchmark: partial-R² of observed covariates
    benchmarks = {}
    other_cols = [c for c in X_cols if c != treatment and c != outcome]
    for col in other_cols:
        r_t = np.corrcoef(res_T, df[col].values[:len(res_T)])[0, 1]
        r_y = np.corrcoef(res_Y, df[col].values[:len(res_Y)])[0, 1]
        benchmarks[col] = (abs(r_t), abs(r_y))

    return r_TU, r_YU_breakdown, breakdown_product, benchmarks, sigma_T, sigma_Y

def robustness_bound(iv_coef, iv_se, breakdown_product):
    """
    Robustness value (RV): the common partial-R² with both T and Y
    needed to reduce the estimate to zero.
    RV = r* such that r* * r* = breakdown_product → r* = sqrt(breakdown_product)
    """
    return np.sqrt(min(breakdown_product, 1.0))


# ══════════════════════════════════════════════════════════════════════════════
# DATASET CONFIGURATIONS
# ══════════════════════════════════════════════════════════════════════════════

CONFIGS = [
    {
        'name':        'causal_direction_iv',
        'df_var':      'data',
        'instruments': ['Z_1', 'Z_2'],
        'treatment':   'X_1',
        'outcome':     'X_2',
        'iv_valid':    True,
        'note':        'Both IVs valid (F=168.7)',
    },
    {
        'name':        'clinical_trial',
        'df_var':      'data_ct',
        'instruments': ['baseline_health'],
        'treatment':   'dosage_mg',
        'outcome':     'health_improvement',
        'iv_valid':    False,
        'note':        'IV failed (F=0.86) — OLS fallback only',
    },
    {
        'name':        'ecommerce — income chain',
        'df_var':      'data_ecom',
        'instruments': None,            # direct OLS only
        'treatment':   'income',
        'outcome':     'purchases',
        'iv_valid':    False,
        'note':        'Direct OLS (no IV needed for income→purchases)',
    },
    {
        'name':        'environment — soil chain',
        'df_var':      'data_env',
        'instruments': ['rainfall_mm'],
        'treatment':   'soil_quality',
        'outcome':     'crop_yield',
        'iv_valid':    True,
        'note':        'rainfall_mm is valid IV (F=859,535)',
    },
    {
        'name':        'marketing — promo chain',
        'df_var':      'data_mkt',
        'instruments': None,            # promo is direct driver
        'treatment':   'promo_spend_k',
        'outcome':     'sales',
        'iv_valid':    False,
        'note':        'Direct OLS (degenerate IV for ad_spend — not used)',
    },
]

# ══════════════════════════════════════════════════════════════════════════════
#  SENSITIVITY ANALYSIS
# ══════════════════════════════════════════════════════════════════════════════

def plot_sensitivity(datasets_dict):
    """
    Imbens partial-R² sensitivity plots for datasets with valid IVs.

    Each plot shows:
      - X-axis: partial correlation of unmeasured confounder U with treatment
      - Y-axis: partial correlation of U with outcome
      - Red curve: the 'breakdown contour' — combinations that would
                   reduce the IV estimate to zero
      - Shaded region: confounders ABOVE the curve overturn the result
      - Diamond markers: where observed covariates sit (benchmark)
      - RV annotation: the robustness value (single r* that breaks down)
    """
    valid_configs = [c for c in CONFIGS if c['iv_valid'] and c['instruments']]

    n = len(valid_configs)
    fig, axes = plt.subplots(1, n, figsize=(7 * n, 5.5))
    if n == 1:
        axes = [axes]
    fig.patch.set_facecolor(C['bg'])
    fig.suptitle('Sensitivity Analysis — How Strong Must Unmeasured Confounding Be\n'
                 'to Overturn the IV Estimate?',
                 fontsize=12, fontweight='bold', color=C['text'], y=1.03)

    print("=" * 70)
    print("SENSITIVITY ANALYSIS RESULTS")
    print("=" * 70)

    for ax, cfg in zip(axes, valid_configs):
        ax.set_facecolor(C['bg'])
        df = datasets_dict[cfg['df_var']]

        iv_coef, iv_se, iv_t, iv_F, _ = run_2sls(
            df, cfg['instruments'], cfg['treatment'], cfg['outcome'])

        r_TU, r_YU_bd, bdp, benchmarks, sigma_T, sigma_Y = sensitivity_partial_r2(
            df, cfg['instruments'], cfg['treatment'], cfg['outcome'],
            iv_coef, iv_se)

        rv = robustness_bound(iv_coef, iv_se, bdp)

        # ── Plot breakdown contour ──
        ax.plot(r_TU, r_YU_bd, color='#A32D2D', lw=2.2, label='Breakdown contour')
        ax.fill_between(r_TU, r_YU_bd, 1.0,
                        alpha=0.10, color='#A32D2D',
                        label='Confounders that overturn result')

        # ── Safe region label ──
        ax.text(0.5, 0.15, 'Result holds\n(safe region)',
                ha='center', fontsize=9, color=C['iv'],
                transform=ax.transAxes,
                bbox=dict(boxstyle='round,pad=0.3', fc='white', ec=C['iv'],
                          lw=0.8, alpha=0.9))
        ax.text(0.5, 0.78, 'Result overturned\n(danger region)',
                ha='center', fontsize=9, color='#A32D2D',
                transform=ax.transAxes,
                bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='#A32D2D',
                          lw=0.8, alpha=0.9))

        # ── Robustness value marker ──
        ax.plot(rv, rv, '*', color='#D85A30', ms=14, zorder=6,
                label=f'Robustness value RV = {rv:.3f}')
        ax.annotate(f'RV = {rv:.3f}',
                    xy=(rv, rv), xytext=(rv + 0.06, rv + 0.06),
                    fontsize=8.5, color='#D85A30',
                    arrowprops=dict(arrowstyle='->', color='#D85A30', lw=1))

        # ── Benchmark: observed covariates ──
        bm_colors = ['#7F77DD', '#1D9E75', '#BA7517', '#639922']
        for j, (col, (rt, ry)) in enumerate(benchmarks.items()):
            color = bm_colors[j % len(bm_colors)]
            ax.plot(rt, ry, 'D', color=color, ms=8, zorder=5)
            ax.text(rt + 0.015, ry + 0.015, col,
                    fontsize=7.5, color=color, fontweight='bold')

        # ── Diagonal (equal confounding) ──
        ax.plot([0, 1], [0, 1], color=C['grid'], lw=0.8, linestyle='--', alpha=0.6)

        # ── Formatting ──
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.set_xlabel('Partial corr. of U with treatment', fontsize=9, color=C['muted'])
        ax.set_ylabel('Partial corr. of U with outcome', fontsize=9, color=C['muted'])
        ax.set_title(f'{cfg["name"]}\n'
                     f'IV estimate: β = {iv_coef:.3f}  (t = {iv_t:.2f})',
                     fontsize=9.5, fontweight='bold', color=C['text'])
        ax.tick_params(colors=C['muted'], labelsize=8)
        ax.legend(fontsize=7.5, loc='upper left', framealpha=0.9)
        for spine in ['top', 'right']:
            ax.spines[spine].set_visible(False)
        for spine in ['bottom', 'left']:
            ax.spines[spine].set_color(C['grid'])
        ax.grid(True, color=C['grid'], alpha=0.4, lw=0.5)

        # ── Print interpretation ──
        print(f"\n{cfg['name'].upper()}")
        print(f"   IV estimate      : β = {iv_coef:.4f}  (t = {iv_t:.2f})")
        print(f"   Breakdown product: r_TU × r_YU must exceed {bdp:.4f}")
        print(f"   Robustness value : RV = {rv:.4f}")
        print(f"   Interpretation   : An unmeasured confounder with partial")
        print(f"                      correlation ≥ {rv:.3f} with BOTH treatment")
        print(f"                      and outcome would reduce β to zero.")
        if benchmarks:
            print(f"   Observed benchmarks:")
            for col, (rt, ry) in benchmarks.items():
                print(f"     {col:20s}: r_treat={rt:.3f}, r_outcome={ry:.3f}  "
                      f"{'⚠ above breakdown!' if rt*ry >= bdp else 'below breakdown'}  ")

    plt.tight_layout()
    plt.savefig('sensitivity_analysis.png', dpi=150, bbox_inches='tight',
                facecolor=C['bg'])
    plt.show()
    print("\nSaved: sensitivity_analysis.png")

# ══════════════════════════════════════════════════════════════════════════════
#  COMBINED SUMMARY FIGURE
# ══════════════════════════════════════════════════════════════════════════════

def plot_bias_magnitude_summary(datasets_dict):
    """
    Horizontal bar chart showing absolute OLS bias (OLS - 2SLS) per dataset.
    Highlights which estimates are most affected by endogeneity.
    Only for datasets where a valid 2SLS estimate exists.
    """
    fig, ax = plt.subplots(figsize=(25, 10))
    fig.patch.set_facecolor(C['bg'])
    ax.set_facecolor(C['bg'])

    labels, biases, pcts, colors_bar = [], [], [], []

    for cfg in CONFIGS:
        if not cfg['instruments']:
            continue
        df = datasets_dict[cfg['df_var']]
        ols_coef, ols_se, _, _ = run_ols(df, cfg['treatment'], cfg['outcome'])
        iv_coef, iv_se, iv_t, iv_F, _ = run_2sls(
            df, cfg['instruments'], cfg['treatment'], cfg['outcome'])

        bias = ols_coef - iv_coef
        pct  = (bias / iv_coef * 100) if iv_coef != 0 else 0

        short = cfg['name'].replace('causal_direction_iv', 'causal_dir_iv')
        labels.append(f"{short}")
        biases.append(bias)
        pcts.append(pct)

        if not cfg['iv_valid']:
            colors_bar.append(C['warn'])
        elif abs(pct) > 10:
            colors_bar.append('#A32D2D')
        elif abs(pct) > 2:
            colors_bar.append(C['ols'])
        else:
            colors_bar.append(C['iv'])

    y_pos = range(len(labels))
    bars = ax.barh(list(y_pos), biases, color=colors_bar, height=0.5,
                   edgecolor='white', linewidth=0.5)

    # Value labels
    for bar, bias, pct in zip(bars, biases, pcts):
        x = bar.get_width()
        ax.text(x + (0.002 if x >= 0 else -0.002),
                bar.get_y() + bar.get_height() / 2,
                f'{bias:+.4f} ({pct:+.1f}%)',
                va='center', ha='left' if x >= 0 else 'right',
                fontsize=12, color=C['text'])

    ax.axvline(0, color=C['text'], lw=1.2)
    ax.set_yticks(list(y_pos))
    ax.set_yticklabels(labels, fontsize=9.5)
    ax.set_xlabel('OLS bias = OLS coefficient − 2SLS coefficient', fontsize=9, color=C['muted'])
    ax.set_title('Endogeneity Bias: How Much Does OLS Over/Underestimate the Causal Effect?',
                 fontsize=11, fontweight='bold', color=C['text'], pad=12)
    ax.tick_params(colors=C['muted'])
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
    for spine in ['bottom', 'left']:
        ax.spines[spine].set_color(C['grid'])
    ax.grid(axis='x', color=C['grid'], alpha=0.4, lw=0.5)

    # Legend
    from matplotlib.patches import Patch
    legend_items = [
        Patch(color=C['iv'],    label='Bias < 2% (OLS reliable)'),
        Patch(color=C['ols'],   label='Bias 2–10% (notable)'),
        Patch(color='#A32D2D',  label='Bias > 10% (OLS misleading)'),
        Patch(color=C['warn'],  label='Weak/degenerate IV (bias unreliable)'),
    ]
    ax.legend(handles=legend_items, fontsize=7.5, loc='lower right', framealpha=0.9)

    plt.tight_layout()
    plt.savefig('bias_magnitude_summary.png', dpi=150, bbox_inches='tight',
                facecolor=C['bg'])
    plt.show()
    print("Saved: bias_magnitude_summary.png")


# ══════════════════════════════════════════════════════════════════════════════
# RUN ALL
# ══════════════════════════════════════════════════════════════════════════════

# Build lookup dict from variable names to actual DataFrames
datasets_dict = {
    'data':      data,
    'data_ct':   data_ct,
    'data_ecom': data_ecom,
    'data_env':  data_env,
    'data_mkt':  data_mkt,
}

print("\n" + "=" * 70)
print("SENSITIVITY ANALYSIS (valid IV datasets only)")
print("=" * 70)
plot_sensitivity(datasets_dict)

print("\n" + "=" * 70)
print("BIAS MAGNITUDE SUMMARY")
print("=" * 70)
plot_bias_magnitude_summary(datasets_dict)


In [ ]:
# ====== run control function analysis to check endogeneity ======
def run_control_function(df, instruments, treatment, outcome, name):
    """
    Control Function Approach (CFA).
    Equivalent to 2SLS but adds a Hausman endogeneity test for free.
    
    Stage 1: regress treatment on instruments → get residuals
    Stage 2: regress outcome on ACTUAL treatment + residuals
    
    Key output: if residual coefficient is significant → endogeneity confirmed
    """
    print(f"\n{'='*55}")
    print(f"  Control Function — {name}")
    print(f"  {instruments} → {treatment} → {outcome}")
    print(f"{'='*55}")

    Z = sm.add_constant(df[instruments])
    X = df[treatment]
    Y = df[outcome]

    # ── Stage 1: treatment on instruments ────────────────
    stage1 = sm.OLS(X, Z).fit()
    residuals = stage1.resid
    X_hat = stage1.fittedvalues

    print(f"\n[Stage 1] {instruments} → {treatment}")
    print(f"  F-stat : {stage1.fvalue:.2f}")
    print(f"  R²     : {stage1.rsquared:.4f}")

    # ── Stage 2: outcome on actual X + residuals ─────────
    # Using ACTUAL X (not X_hat) — this is what makes CFA different
    X_with_resid = sm.add_constant(
        pd.DataFrame({treatment: X, 'resid_stage1': residuals})
    )
    stage2 = sm.OLS(Y, X_with_resid).fit()

    coef_X     = stage2.params[treatment]
    coef_resid = stage2.params['resid_stage1']
    se_resid   = stage2.bse['resid_stage1']
    t_resid    = stage2.tvalues['resid_stage1']
    p_resid    = stage2.pvalues['resid_stage1']

    print(f"\n[Stage 2] {treatment} + residuals → {outcome}")
    print(f"  β ({treatment})   : {coef_X:.4f}  ← causal effect (same as 2SLS)")
    print(f"  β (residuals)       : {coef_resid:.4f}")
    print(f"  SE (residuals)      : {se_resid:.4f}")
    print(f"  t (residuals)       : {t_resid:.4f}")
    print(f"  p (residuals)       : {p_resid:.4f}")

    # ── Hausman endogeneity test interpretation ───────────
    print(f"\n[Hausman Endogeneity Test]")
    if p_resid < 0.05:
        print(f"  Residual coef significant (p={p_resid:.4f})")
        print(f"     → {treatment} IS endogenous")
        print(f"     → OLS was biased. IV / 2SLS was necessary.")
    else:
        print(f"  Residual coef NOT significant (p={p_resid:.4f})")
        print(f"     → {treatment} may be exogenous")
        print(f"     → OLS and 2SLS estimates should be similar.")

    # ── Compare CFA vs 2SLS vs OLS ───────────────────────
    # OLS for comparison
    ols_model = sm.OLS(Y, sm.add_constant(X)).fit()
    ols_coef  = ols_model.params[1]

    # 2SLS for comparison (manual)
    X_hat_c  = sm.add_constant(X_hat)
    ss       = sm.OLS(Y, X_hat_c).fit()
    beta     = ss.params.values
    X_act_c  = sm.add_constant(X)
    resid_c  = Y.values - X_act_c.values @ beta
    sigma2   = np.sum(resid_c**2) / (len(Y) - 2)
    XtX_inv  = np.linalg.inv(X_hat_c.values.T @ X_hat_c.values)
    iv_se    = np.sqrt(np.diag(sigma2 * XtX_inv))[1]
    iv_coef  = beta[1]

    print(f"\n[Coefficient Comparison]")
    print(f"  OLS estimate : {ols_coef:.4f}")
    print(f"  2SLS estimate: {iv_coef:.4f}  (SE={iv_se:.4f})")
    print(f"  CFA estimate : {coef_X:.4f}   ← should match 2SLS exactly")
    print(f"  OLS bias     : {ols_coef - iv_coef:+.4f}")

    return {
        'Dataset':         name,
        'OLS β':           round(ols_coef, 4),
        '2SLS β':          round(iv_coef, 4),
        'CFA β':           round(coef_X, 4),
        'Residual coef':   round(coef_resid, 4),
        'Residual p':      round(p_resid, 4),
        'Endogenous?':     'Yes' if p_resid < 0.05 else 'No',
        'OLS bias':        round(ols_coef - iv_coef, 4),
    }

# ── Run on valid IV datasets only ─────────────────────────────────────────────
print("█" * 55)
print("█  CONTROL FUNCTION APPROACH — ENDOGENEITY TESTS")
print("█" * 55)

cfa_results = []

cfa_results.append(run_control_function(
    df          = data,
    instruments = ['Z_1', 'Z_2'],
    treatment   = 'X_1',
    outcome     = 'X_2',
    name        = 'causal_direction_iv',
))

cfa_results.append(run_control_function(
    df          = data_env,
    instruments = ['rainfall_mm'],
    treatment   = 'soil_quality',
    outcome     = 'crop_yield',
    name        = 'environment — soil chain',
))

# ── Summary table ─────────────────────────────────────────────────────────────
print(f"\n{'='*55}")
print("  SUMMARY")
print(f"{'='*55}")
df_cfa = pd.DataFrame(cfa_results)
print(df_cfa.to_string(index=False))

## Comparative Analysis

Compare the performance of the model(s) against the baseline model. Discuss any improvements or setbacks and the reasons behind them.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  OLS vs 2SLS BIAS COMPARISON
# ══════════════════════════════════════════════════════════════════════════════

def plot_ols_vs_2sls(datasets_dict):
    """
    Side-by-side coefficient plots for all datasets.
    Each panel shows OLS vs 2SLS estimate with 95% CI bars.
    """
    n = len(CONFIGS)
    fig, axes = plt.subplots(1, n, figsize=(18, 5), sharey=False)
    fig.patch.set_facecolor(C['bg'])
    fig.suptitle('OLS vs 2SLS Estimates — Bias Comparison Across Datasets',
                 fontsize=13, fontweight='bold', color=C['text'], y=1.02)

    results = []

    for i, cfg in enumerate(CONFIGS):
        ax = axes[i]
        ax.set_facecolor(C['bg'])
        df = datasets_dict[cfg['df_var']]

        # OLS estimate
        ols_coef, ols_se, ols_p, ols_r2 = run_ols(df, cfg['treatment'], cfg['outcome'])

        # 2SLS estimate (only if instruments exist)
        if cfg['instruments'] is not None:
            iv_coef, iv_se, iv_t, iv_F, _ = run_2sls(
                df, cfg['instruments'], cfg['treatment'], cfg['outcome'])
            has_iv = True
        else:
            iv_coef = iv_se = iv_t = iv_F = None
            has_iv = False

        # ── Plot ──
        positions = [0.3, 0.7] if has_iv else [0.5]
        labels    = ['OLS', '2SLS'] if has_iv else ['OLS']
        coefs     = [ols_coef, iv_coef] if has_iv else [ols_coef]
        ses       = [ols_se, iv_se] if has_iv else [ols_se]
        colors    = [C['ols'], C['iv']] if has_iv else [C['ols']]

        for pos, lbl, coef, se, color in zip(positions, labels, coefs, ses, colors):
            ci_lo = coef - 1.96 * se
            ci_hi = coef + 1.96 * se
            ax.plot([pos, pos], [ci_lo, ci_hi], color=color, lw=2.5, solid_capstyle='round')
            ax.plot(pos, coef, 'o', color=color, ms=9, zorder=5)
            ax.text(pos, ci_hi + 0.02 * (ci_hi - ci_lo + 0.1),
                    f'{coef:.3f}', ha='center', fontsize=8.5,
                    color=color, fontweight='bold')

        # Bias annotation
        if has_iv and iv_coef is not None:
            bias = ols_coef - iv_coef
            pct  = (bias / iv_coef) * 100 if iv_coef != 0 else float('nan')
            bias_color = '#A32D2D' if abs(pct) > 5 else C['muted']
            ax.annotate('', xy=(0.7, iv_coef), xytext=(0.3, ols_coef),
                        arrowprops=dict(arrowstyle='<->', color=bias_color,
                                        lw=1.2, linestyle='dashed'))
            mid_y = (ols_coef + iv_coef) / 2
            ax.text(0.5, mid_y, f'bias\n{bias:+.3f}\n({pct:+.1f}%)',
                    ha='center', va='center', fontsize=7.5,
                    color=bias_color,
                    bbox=dict(boxstyle='round,pad=0.25', fc='white', ec=bias_color,
                              lw=0.7, alpha=0.9))

        # Zero line
        ax.axhline(0, color=C['grid'], lw=0.8, linestyle=':')

        # IV validity badge
        if not cfg['iv_valid'] and has_iv:
            badge_txt = 'weak IV'
            badge_col = C['warn']
        elif not has_iv:
            badge_txt = 'OLS only'
            badge_col = C['fail']
        else:
            badge_txt = 'IV valid'
            badge_col = C['iv']

        ax.text(0.5, 1.02, badge_txt, transform=ax.transAxes,
                ha='center', fontsize=7.5, color=badge_col,
                bbox=dict(boxstyle='round,pad=0.3', fc='white', ec=badge_col, lw=0.8))

        # Labels
        ax.set_title(f'{cfg["name"]}',
                     fontsize=8.5, fontweight='bold', color=C['text'], pad=22)
        ax.set_xticks(positions)
        ax.set_xticklabels(labels, fontsize=9)
        ax.set_xlim(0, 1)
        ax.tick_params(colors=C['muted'])
        ax.set_ylabel('Coefficient estimate', fontsize=8, color=C['muted'])
        for spine in ['top', 'right']:
            ax.spines[spine].set_visible(False)
        for spine in ['bottom', 'left']:
            ax.spines[spine].set_color(C['grid'])

        note_txt = f'F={iv_F:.0f}' if iv_F else ''
        ax.text(0.5, -0.18, cfg['note'], transform=ax.transAxes,
                ha='center', fontsize=7, color=C['muted'], style='italic')

        results.append({
            'Dataset': cfg['name'],
            'OLS coef': round(ols_coef, 4),
            'OLS SE':   round(ols_se, 4),
            '2SLS coef': round(iv_coef, 4) if iv_coef else 'N/A',
            '2SLS SE':  round(iv_se, 4) if iv_se else 'N/A',
            'Bias (OLS-2SLS)': round(ols_coef - iv_coef, 4) if iv_coef else 'N/A',
            'Bias %': f'{((ols_coef-iv_coef)/iv_coef*100):+.1f}%' if iv_coef else 'N/A',
            'IV valid': cfg['iv_valid'],
        })

    # Legend
    handles = [
        plt.Line2D([0], [0], marker='o', color=C['ols'], lw=2, ms=7, label='OLS'),
        plt.Line2D([0], [0], marker='o', color=C['iv'],  lw=2, ms=7, label='2SLS (IV)'),
    ]
    fig.legend(handles=handles, loc='lower center', ncol=2,
               fontsize=9, frameon=False, bbox_to_anchor=(0.5, -0.12))

    plt.tight_layout()
    plt.savefig('ols_vs_2sls_comparison.png', dpi=150, bbox_inches='tight',
                facecolor=C['bg'])
    plt.show()
    print("Saved: ols_vs_2sls_comparison.png\n")

    # Print summary table
    df_results = pd.DataFrame(results)
    print("=" * 70)
    print("OLS vs 2SLS SUMMARY TABLE")
    print("=" * 70)
    print(df_results.to_string(index=False))
    print()
    return df_results

# Plot OLS vs 2SLS comparison
print("=" * 70)
print("PART 1: OLS vs 2SLS COMPARISON")
print("=" * 70)
summary_df = plot_ols_vs_2sls(datasets_dict)

**Discussion:**